# Тест лёгкой генерации изображений: Stable Diffusion 1.5
Одна картинка 512×512. Без Gradio, веб-сервера и фонового цикла. После сохранения PNG процесс завершается.

In [ ]:
import os, gc, torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not available')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1024**3:.1f} GiB')
os.makedirs('/kaggle/working', exist_ok=True)


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers', 'transformers', 'accelerate', 'safetensors'])
from diffusers import StableDiffusionPipeline
pipe = None
try:
    pipe = StableDiffusionPipeline.from_pretrained(
        'stable-diffusion-v1-5/stable-diffusion-v1-5',
        torch_dtype=torch.float16,
        variant='fp16',
        use_safetensors=True,
        safety_checker=None,
        requires_safety_checker=False,
    ).to('cuda:0')
    pipe.enable_attention_slicing()
    prompt = 'A charming fluffy ginger cat sitting on a windowsill, warm morning sunlight, detailed realistic fur, cozy elegant room, professional photography'
    result = pipe(prompt, height=512, width=512, num_inference_steps=18, guidance_scale=7.0)
    image = result.images[0]
    output_path = '/kaggle/working/sd15-test.png'
    image.save(output_path)
    print('IMAGE_READY:', output_path, 'size:', os.path.getsize(output_path))
finally:
    if pipe is not None:
        del pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    print('GPU memory cache cleared. Notebook will now finish; no server was started.')
